# XGBoost

| Feature / Concept | Purpose | Description |
|-------------------|----------|-------------|
| Gradient + Hessian Optimization | Better split quality | Uses first-order (gradient) and second-order (Hessian) information to compute optimal splits and leaf values, improving convergence stability and accuracy. |
| Regularized Objective | Prevent overfitting | Optimizes loss + complexity penalty, explicitly controlling tree complexity via L1/L2 regularization and leaf structure penalties. |
| Tree Pruning (Post-pruning) | Remove weak splits | Trees are grown and then pruned backward using gain-based criteria (gamma threshold), ensuring only useful splits remain. |
| Gamma (`min_split_loss`) | Split filtering | Minimum loss reduction required to make a split; higher gamma makes model more conservative. |
| Subsample | Row sampling | Randomly samples a fraction of training rows per tree to reduce variance and overfitting. |
| Colsample_bytree | Feature sampling | Randomly samples features per tree, reducing correlation between trees and improving generalization. |
| Max Depth | Controls complexity | Limits depth of each tree to prevent overfitting; deeper trees capture more interactions but risk noise fitting. |
| Learning Rate (eta) | Step size control | Scales contribution of each tree; lower values require more trees but improve generalization. |
| Missing Value Handling | Native sparsity support | Learns optimal direction (left/right) for missing values during split optimization instead of requiring imputation. |
| Sparsity Awareness | Efficient computation | Optimized for sparse matrices and zero-heavy datasets, skipping unnecessary computations. |
| L1 Regularization (alpha) | Feature sparsity | Encourages sparse leaf weights, potentially zeroing weak contributions. |
| L2 Regularization (lambda) | Leaf shrinkage | Penalizes large leaf values, making predictions smoother and reducing overconfidence. |
| Early Stopping | Prevent overfitting | Stops training when validation metric stops improving after a defined number of iterations. |
| Tree Booster (`gbtree`, `dart`) | Model variant | `gbtree` is standard boosting; `dart` introduces dropout-style tree regularization for additional robustness. |
| Parallel Split Finding | Training speed | Efficient histogram/sorting optimizations allow faster split computation on large datasets. |

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

In [2]:
# existing dataset : fraud (some columsn are not useful for the training)
df = pd.read_csv('data/development_data_preprocessed.csv')
drop_cols = ['fraud_tx_count_next_1h','history_first_tx_time','history_last_tx_time',
       'first_fraud_time_next_1h', 'fraud_amount_next_1h','last_tx_site_id',
       'minutes_to_first_fraud_next_1h','card_id', 'run_time', 'feature_cutoff_time', 'decision_time']

df = df[[i for i in df.columns if i not in drop_cols]]

In [3]:
df.shape

(19555, 79)

In [4]:
df.columns

Index(['history_tx_count_total', 'history_spend_total',
       'history_quantity_total', 'history_fraud_tx_count_total', 'tx_count_1h',
       'spend_1h', 'quantity_1h', 'tx_count_24h', 'spend_24h', 'quantity_24h',
       'avg_amount_24h', 'max_amount_24h', 'tx_count_7d', 'spend_7d',
       'quantity_7d', 'avg_amount_7d', 'tx_count_30d', 'spend_30d',
       'quantity_30d', 'avg_amount_30d', 'max_amount_30d', 'std_amount_30d',
       'avg_quantity_30d', 'cross_border_count_24h', 'cross_border_count_30d',
       'country_change_count_30d', 'quick_repeat_count_30d',
       'impossible_travel_150_count_30d', 'impossible_travel_300_count_30d',
       'night_tx_count_30d', 'weekend_tx_count_30d', 'magstripe_tx_count_30d',
       'mobile_tx_count_30d', 'chip_tx_count_30d', 'max_distance_km_30d',
       'avg_distance_km_30d', 'avg_gap_minutes_30d',
       'hours_since_first_transaction', 'minutes_since_last_tx',
       'hours_since_last_cross_border_tx', 'hours_since_last_fraud_tx',
       'la

In [5]:
df.isna().sum()[df.isna().sum()>0] # no nulls

Series([], dtype: int64)

In [6]:
# finding the categorical features: object type : converting to category type 
string_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

In [7]:
for i in string_cols:
    df[i] = df[i].astype("category")

In [8]:
df.select_dtypes(include=['category']).columns.tolist()

['last_tx_site_country',
 'last_tx_product_description',
 'last_tx_processing_method']

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop("target_next_1h", axis=1)
y = df["target_next_1h"]

# Test Split
X_temp, X_test, y_temp, y_test = train_test_split(
    X,y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Validation Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp
)

print(len(X_train))
print(len(X_valid))
print(len(X_test))

13687
2934
2934


In [10]:
dtrain = xgb.DMatrix(X_train, label=y_train,enable_categorical=True)
dvalid = xgb.DMatrix(X_valid, label=y_valid,enable_categorical=True)
dtest  = xgb.DMatrix(X_test, label=y_test,enable_categorical=True)
# we dont need to manually set enable_categorical=True if we are using pandas categoies

In [11]:
mono_constraints = {
    X_train.columns[0]: 1,
    X_train.columns[1]: -1
}

In [12]:
mono_constraints

{'history_tx_count_total': 1, 'history_spend_total': -1}

In [13]:
params = {
    "objective": "binary:logistic",   # binary classification with probability output
    "eval_metric": "auc",             # ranking quality metric (robust for imbalance)
    "eta": 0.03,                      # learning rate (smaller = better generalization)
    "max_depth": 8,                   # maximum tree depth (controls complexity)
    "min_child_weight": 1,            # minimum hessian sum in a child (prevents small noisy leaves)
    "gamma": 0.1,                     # minimum loss reduction required to split (regularization strength)
    "subsample": 0.8,                 # row sampling per tree (variance reduction)
    "colsample_bytree": 0.8,          # feature sampling per tree (decorrelates trees)
    "lambda": 1.0,                    # L2 regularization on leaf weights (shrinks leaf outputs)
    "alpha": 0.5,                     # L1 regularization on leaf weights (induces sparsity)
    "tree_method": "hist",            # histogram-based split finding (fast + scalable)
    "scale_pos_weight": 10,           # handles class imbalance (fraud-style datasets)
    "monotone_constraints": mono_constraints,  # enforce feature monotonicity constraints
    "seed": 42                        # reproducibility
}

In [14]:
evals = [(dtrain, "train"), (dvalid, "valid")]

In [15]:
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,          # number of boosting iterations (trees)
    evals=evals,
    early_stopping_rounds=50,      # stops if validation metric doesn't improve
    verbose_eval=50                # logs every 50 iterations
)

[0]	train-auc:0.99623	valid-auc:0.99589
[50]	train-auc:0.99997	valid-auc:0.99645
[75]	train-auc:0.99999	valid-auc:0.99633
